# 01 — Data Collection

Collect lap-by-lap race data from FastF1 for the 2022–2025 seasons, merge with track metadata, and save to `data/raw/`.

**Rate-limit note**: The FastF1 API allows ~500 calls/hour. Each race loads ~8 endpoints, so we can process ~60 races per hour. A sleep is added between races. Already-cached races are instant. If you hit the limit mid-run, re-run the cell — completed seasons are saved and skipped automatically.

## Setup

In [1]:
import time
import warnings
from pathlib import Path

import fastf1
import pandas as pd

warnings.filterwarnings("ignore")

fastf1.Cache.enable_cache("../f1_cache")

RAW_DIR = Path("../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Seconds to sleep between fetching each race.
# Set to 0 if all data is already cached locally.
SLEEP_BETWEEN_RACES = 10

## Load track metadata

In [2]:
track_metadata = pd.read_csv("../data/track_metadata.csv")

print(f"Circuits in metadata: {len(track_metadata)}")
assert track_metadata.isna().sum().sum() == 0, "Track metadata has missing values."
track_metadata.head()

Circuits in metadata: 25


,circuit_key,grand_prix,circuit_name,city,country,circuit_type,track_length_km,num_corners,num_drs_zones,race_laps,...,direction,elevation_change_m,overtaking_difficulty,tyre_stress,in_2022,in_2023,in_2024,notes,fastf1_location,fastf1_event
0,BAH,Bahrain,Bahrain International Circuit,Sakhir,Bahrain,Permanent,5.412,15,2,57,...,Clockwise,37,Low,High,1,1,1,Night race; desert abrasive surface; high tyre...,Bahrain,Bahrain Grand Prix
1,SAU,Saudi Arabia,Jeddah Corniche Circuit,Jeddah,Saudi Arabia,Street,6.174,27,3,50,...,Clockwise,12,Medium,Medium,1,1,1,Fastest street circuit; very high speed; limit...,Jeddah,Saudi Arabian Grand Prix
2,AUS,Australia,Albert Park Grand Prix Circuit,Melbourne,Australia,Semi-Permanent,5.278,14,4,58,...,Clockwise,35,Medium,Low,1,1,1,Semi-street; fastest pit lane passage on calen...,Melbourne,Australian Grand Prix
3,EMI,Emilia Romagna,Autodromo Enzo e Dino Ferrari,Imola,Italy,Permanent,4.909,19,1,63,...,Clockwise,28,Very High,Medium,1,1,0,Limited overtaking; tight layout; no 2023 race...,Imola,Emilia Romagna Grand Prix
4,MIA,Miami,Miami International Autodrome,Miami,USA,Semi-Permanent,5.412,19,3,57,...,Clockwise,4,Medium,Medium,1,1,1,Stadium complex layout; high humidity; surface...,Miami Gardens,Miami Grand Prix


## Helper — load one race

Loads a race session, attaches `EventName` and `Season`, and merges with track metadata using `fastf1_event` as the join key. Track names are intentionally avoided as join keys because circuit layouts and naming can change across seasons.

In [3]:
METADATA_COLS = [
    "circuit_key",
    "track_length_km",
    "num_corners",
    "num_drs_zones",
    "race_laps",
    "race_distance_km",
    "pit_lane_speed_limit_kmh",
    "pit_loss_delta_s",
    "circuit_type",
    "direction",
    "elevation_change_m",
    "overtaking_difficulty",
    "tyre_stress",
]


def load_race(year: int, event_name: str) -> pd.DataFrame | None:
    """Load one race and return laps merged with track metadata.

    Returns None if the race cannot be loaded (e.g. pre-season testing,
    sprint-only events, or missing data).
    """
    try:
        session = fastf1.get_session(year, event_name, "R")
        session.load(telemetry=False, weather=True, messages=False)
    except Exception as exc:
        print(f"  [SKIP] {year} {event_name}: {exc}")
        return None

    laps = session.laps.copy()
    if laps.empty:
        print(f"  [SKIP] {year} {event_name}: no lap data")
        return None

    laps["EventName"] = session.event["EventName"]
    laps["Season"] = year

    laps = laps.merge(
        track_metadata,
        left_on="EventName",
        right_on="fastf1_event",
        how="left",
        validate="many_to_one",
    )

    missing = laps[METADATA_COLS].isna().sum().sum()
    if missing > 0:
        print(f"  [WARN] {year} {event_name}: {missing} metadata NaNs — event not in track_metadata?")

    laps = laps.drop(columns=["fastf1_location", "fastf1_event"], errors="ignore")
    return laps

## Smoke-test — one race

Verify the helper works on a single known-good race before running the full loop.

In [4]:
test_laps = load_race(2022, "Bahrain")

print(f"Rows: {len(test_laps)}, Columns: {len(test_laps.columns)}")
print(f"circuit_key sample: {test_laps['circuit_key'].iloc[0]}")
test_laps[["Driver", "LapNumber", "Compound", "TyreLife", "Position", "circuit_key", "pit_loss_delta_s"]].head()

core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Driver 16 completed the race distance 00:00.050000 before the recorded end of the session.
req            INFO 	Using cached data for weather_data
core           INFO 	Finished loading data for 20 drivers: ['16', '55', '44', '63', '20', '77', '31', '22', '14', '24', '47', '18', '23', '3', '4', '6', '27', '11', '1', '10']


Rows: 1125, Columns: 54
circuit_key sample: BAH


,Driver,LapNumber,Compound,TyreLife,Position,circuit_key,pit_loss_delta_s
0,VER,1.0,SOFT,4.0,2.0,BAH,22.0
1,VER,2.0,SOFT,5.0,2.0,BAH,22.0
2,VER,3.0,SOFT,6.0,2.0,BAH,22.0
3,VER,4.0,SOFT,7.0,2.0,BAH,22.0
4,VER,5.0,SOFT,8.0,2.0,BAH,22.0


## Collect all seasons

Loop over 2022–2025. Each completed season is saved to a CSV checkpoint immediately. On re-run, existing checkpoints are loaded from disk rather than fetched again.

In [5]:
YEARS = [2022, 2023, 2024, 2025]

season_frames = []

for year in YEARS:
    checkpoint = RAW_DIR / f"season_{year}.csv"

    # Load from checkpoint if already collected.
    if checkpoint.exists():
        print(f"\n=== {year}: loading from checkpoint ===")
        df = pd.read_csv(checkpoint, low_memory=False)
        print(f"  {len(df)} rows, {df['EventName'].nunique()} events")
        season_frames.append(df)
        continue

    print(f"\n=== {year}: fetching from FastF1 ===")

    schedule = fastf1.get_event_schedule(year, include_testing=False)
    events = schedule["EventName"].tolist()

    race_frames = []
    for i, event in enumerate(events):
        print(f"  [{i+1}/{len(events)}] {event}")
        df = load_race(year, event)
        if df is not None:
            race_frames.append(df)

        # Sleep to stay within the 500 calls/hr rate limit.
        # Cached races do not hit the API so the sleep is conservative.
        if SLEEP_BETWEEN_RACES > 0:
            time.sleep(SLEEP_BETWEEN_RACES)

    if not race_frames:
        print(f"  [WARN] No data collected for {year}")
        continue

    season_df = pd.concat(race_frames, ignore_index=True)
    season_df.to_csv(checkpoint, index=False)
    print(f"  Saved {len(season_df)} rows → {checkpoint}")
    season_frames.append(season_df)

print("\nDone.")

core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...



=== 2022: fetching from FastF1 ===
  [1/22] Bahrain Grand Prix


core        WARNING 	Driver 16 completed the race distance 00:00.050000 before the recorded end of the session.
req            INFO 	Using cached data for weather_data
core           INFO 	Finished loading data for 20 drivers: ['16', '55', '44', '63', '20', '77', '31', '22', '14', '24', '47', '18', '23', '3', '4', '6', '27', '11', '1', '10']
core           INFO 	Loading data for Saudi Arabian Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info


  [2/22] Saudi Arabian Grand Prix


req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	No lap data for driver 22
core        WARNING 	Failed to perform lap accuracy check - all laps marked as inaccurate (driver 22)
core        WARNING 	Failed to perform lap accuracy check - all laps marked as inaccurate (driver 47)
req            INFO 	Using cached data for weather_data
core           INFO 	Finished loading data for 20 drivers: ['1', '16', '55', '11', '63', '31', '4', '10', '20', '44', '24', '27', '18', '23', '77', '14', '3', '6', '22', '47']
core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached dat

...

  [23/24] Qatar Grand Prix


req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
req            INFO 	Data has been written to cache!

  [24/24] Abu Dhabi Grand Prix


req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...


  [SKIP] 2025 Abu Dhabi Grand Prix: any API: 500 calls/h
  Saved 25533 rows → ../data/raw/season_2025.csv

Done.


## Combine and save

In [6]:
all_laps = pd.concat(season_frames, ignore_index=True)

out_path = RAW_DIR / "all_laps.csv"
all_laps.to_csv(out_path, index=False)

print(f"Total rows   : {len(all_laps):,}")
print(f"Total columns: {len(all_laps.columns)}")
print(f"Seasons      : {sorted(all_laps['Season'].unique())}")
print(f"Events       : {all_laps['EventName'].nunique()}")
print(f"Saved to     : {out_path}")

## Quick validation

In [7]:
# Rows per season
all_laps.groupby("Season").agg(
    events=("EventName", "nunique"),
    laps=("LapNumber", "count"),
)

Total rows   : 100,134
Total columns: 54
Seasons      : [np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Events       : 25
Saved to     : ../data/raw/all_laps.csv


In [8]:
# Check metadata merge quality — count NaNs in key metadata columns
all_laps[METADATA_COLS].isna().sum()

Total rows   : 100,134
Total columns: 54
Seasons      : [np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Events       : 25
Saved to     : ../data/raw/all_laps.csv


In [9]:
# Verify compound values look clean
all_laps["Compound"].value_counts()

Compound
HARD            43651
MEDIUM          36560
SOFT            12737
INTERMEDIATE     5945
WET               461
None              426
nan               354
Name: count, dtype: int64

In [10]:
# Circuit keys present across seasons
all_laps.groupby(["Season", "circuit_key"]).size().unstack(fill_value=0)

circuit_key,ABU,AUS,AUT,AZE,BAH,BEL,BRA,CAN,CHN,EMI,...,JPN,LAS,MEX,MIA,MON,NED,QAT,SAU,SGP,USA
Season,,,,,,,,,,,,,,,,,,,,,
2022,1117,1045,1324,891,1125,792,1259,1264,0,1132,...,507,0,1379,1057,1179,1392,0,820,945,992
2023,1157,1003,1354,962,1056,816,1108,1317,0,0,...,880,946,1282,1138,1515,1343,1006,943,1088,1014
2024,1035,998,1405,973,1129,841,1134,1272,1032,1238,...,907,938,1215,1111,1237,1426,943,901,1177,1059
2025,0,927,1126,968,1128,879,1251,1349,1065,1207,...,1059,886,1263,1005,1425,1364,1067,898,1229,1067
